# 02 — Logistic Regression on TF-IDF

5-fold GroupKFold with per-fold train/val metrics logged to W&B. Saves OOF and test predictions to `../predictions/` for the inference notebook to load.

In [1]:
import subprocess, sys
def _p(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for pkg in ['wandb', 'pyarrow']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        _p(pkg)

In [2]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
    OUTPUT_DIR = Path("/kaggle/working")
    PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
    FEATURES_DIR = OUTPUT_DIR / "features"
    FIGURES_DIR = OUTPUT_DIR / "figures"
else:
    ROOT = Path.cwd()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent
    DATA_DIR = ROOT / "data"
    PREDICTIONS_DIR = ROOT / "predictions"
    FEATURES_DIR = ROOT / "features"
    FIGURES_DIR = ROOT / "figures"

for p in [PREDICTIONS_DIR, FEATURES_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"IS_KAGGLE={IS_KAGGLE}")
print(f"DATA_DIR={DATA_DIR}")
print(f"PREDICTIONS_DIR={PREDICTIONS_DIR}")
print(f"FEATURES_DIR={FEATURES_DIR}")
print(f"FIGURES_DIR={FIGURES_DIR}")

IS_KAGGLE=False
DATA_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\data
PREDICTIONS_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\predictions
FEATURES_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\features
FIGURES_DIR=c:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\figures


In [3]:
import wandb
WANDB_ENTITY = "23f2001849-dl-genai-project"
WANDB_PROJECT = "23f2001849-t22026"
USE_WANDB = False
try:
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    else:
        wandb_key = os.environ.get("WANDB_API_KEY", "").strip()
    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        USE_WANDB = True
        print("W&B login ok")
    else:
        print("W&B key not found, running without logging")
except Exception as e:
    print(f"W&B login failed: {e}. Running without logging.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Krishna\_netrc


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login ok


In [4]:
import json
import time
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, log_loss)

SEED = 42
OPTIONS = ['A', 'B', 'C', 'D', 'E']
NUM_CHOICES = 5
FOLDS = 5

In [5]:
train_long = pd.read_parquet(FEATURES_DIR / 'train_long.parquet')
test_long = pd.read_parquet(FEATURES_DIR / 'test_long.parquet')
train_meta = pd.read_parquet(FEATURES_DIR / 'train_meta.parquet')
test_meta = pd.read_parquet(FEATURES_DIR / 'test_meta.parquet')
print(f'train_long: {train_long.shape}, test_long: {test_long.shape}')

train_long: (10000, 138), test_long: (2500, 136)


## MAP@3 helpers

In [6]:
def map3_score(pred_letters, true_letters):
    s = 0.0
    for pred, true in zip(pred_letters, true_letters):
        for k, L in enumerate(pred[:3]):
            if L == true:
                s += 1.0 / (k + 1)
                break
    return s / len(true_letters)

def scores_to_top3(grid):
    order = np.argsort(-grid, axis=1)
    return [[OPTIONS[i] for i in row[:3]] for row in order]

def long_to_grid(long_df, scores, ids_order):
    id_to_row = {q: i for i, q in enumerate(ids_order)}
    grid = np.zeros((len(ids_order), NUM_CHOICES))
    for i in range(len(long_df)):
        row = long_df.iloc[i]
        grid[id_to_row[row['id']], OPTIONS.index(row['letter'])] = scores[i]
    return grid

def softmax_per_q(grid):
    e = np.exp(grid - grid.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

## TF-IDF matrix

In [7]:
tr_texts = [f'{p} [SEP] {o}' for p, o in zip(train_long['prompt'].astype(str),
                                                train_long['option'].astype(str))]
te_texts = [f'{p} [SEP] {o}' for p, o in zip(test_long['prompt'].astype(str),
                                                test_long['option'].astype(str))]

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60_000,
                       sublinear_tf=True, strip_accents='unicode')
vec.fit(tr_texts + te_texts)
X_tr = vec.transform(tr_texts)
X_te = vec.transform(te_texts)
y = train_long['is_correct'].astype(int).values
folds = train_long['fold'].values
print(f'X_tr: {X_tr.shape}, X_te: {X_te.shape}')

X_tr: (10000, 13110), X_te: (2500, 13110)


## 5-fold training with per-fold metrics

In [8]:
if USE_WANDB:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name='lr',
                     reinit=True, tags=['model:lr', 'features:tfidf'],
                     config={'model': 'logistic_regression', 'C': 1.0,
                              'solver': 'liblinear', 'folds': FOLDS,
                              'features': 'tfidf_word_1-2_60k'},
                     settings=wandb.Settings(_disable_stats=True))

true_letters = train_meta['answer'].tolist()
true_idx = np.array([OPTIONS.index(a) for a in true_letters])
true_ids_order = train_meta['id'].tolist()

oof_lr = np.zeros(len(train_long))
fold_metrics = []
t0 = time.time()

for f in range(FOLDS):
    tr_mask = folds != f
    va_mask = folds == f
    clf = LogisticRegression(C=1.0, max_iter=1000, solver='liblinear',
                              random_state=SEED)
    clf.fit(X_tr[tr_mask], y[tr_mask])
    val_prob = clf.predict_proba(X_tr[va_mask])[:, 1]
    tr_prob = clf.predict_proba(X_tr[tr_mask])[:, 1]
    oof_lr[va_mask] = val_prob

    val_pred = (val_prob > 0.5).astype(int)
    tr_pred = (tr_prob > 0.5).astype(int)

    m = {
        'fold': f,
        'train_loss': log_loss(y[tr_mask], tr_prob),
        'val_loss': log_loss(y[va_mask], val_prob),
        'train_accuracy': accuracy_score(y[tr_mask], tr_pred),
        'train_precision': precision_score(y[tr_mask], tr_pred, zero_division=0),
        'train_recall': recall_score(y[tr_mask], tr_pred, zero_division=0),
        'train_f1': f1_score(y[tr_mask], tr_pred, zero_division=0),
        'val_accuracy': accuracy_score(y[va_mask], val_pred),
        'val_precision': precision_score(y[va_mask], val_pred, zero_division=0),
        'val_recall': recall_score(y[va_mask], val_pred, zero_division=0),
        'val_f1': f1_score(y[va_mask], val_pred, zero_division=0),
    }
    # 5-way MC top1 and MAP@3 on this fold's questions
    val_grid = long_to_grid(train_long[va_mask], val_prob,
                             train_meta.loc[train_meta['fold'] == f, 'id'].tolist())
    top3_va = scores_to_top3(val_grid)
    true_va = train_meta.loc[train_meta['fold'] == f, 'answer'].tolist()
    m['val_top1_acc'] = float(np.mean([t[0] == a for t, a in zip(top3_va, true_va)]))
    m['val_map3'] = map3_score(top3_va, true_va)
    fold_metrics.append(m)

    if USE_WANDB:
        wandb.log({f'fold_{f}/{k}': v for k, v in m.items() if k != 'fold'}, step=f)
    print(f'fold {f}: val_acc {m["val_accuracy"]:.4f}  val_map3 {m["val_map3"]:.4f}')

elapsed = time.time() - t0
print(f'total training time: {elapsed:.1f}s')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


fold 0: val_acc 0.8035  val_map3 0.5171
fold 1: val_acc 0.8040  val_map3 0.6588
fold 2: val_acc 0.8025  val_map3 0.5775
fold 3: val_acc 0.8025  val_map3 0.4329
fold 4: val_acc 0.8015  val_map3 0.5100
total training time: 4.2s


## OOF-level final metrics

In [9]:
oof_pred = (oof_lr > 0.5).astype(int)
grid_lr = long_to_grid(train_long, oof_lr, true_ids_order)
top3 = scores_to_top3(grid_lr)

final_metrics = {
    'accuracy': accuracy_score(y, oof_pred),
    'precision': precision_score(y, oof_pred, zero_division=0),
    'recall': recall_score(y, oof_pred, zero_division=0),
    'f1': f1_score(y, oof_pred, zero_division=0),
    'log_loss': log_loss(y, oof_lr),
    'top1_acc': float(np.mean([t[0] == a for t, a in zip(top3, true_letters)])),
    'map3': map3_score(top3, true_letters),
    'training_time_sec': elapsed,
}
print('LR OOF metrics:')
for k, v in final_metrics.items():
    print(f'  {k}: {v:.4f}')

if USE_WANDB:
    wandb.log({f'final/{k}': v for k, v in final_metrics.items()})
    fold_df = pd.DataFrame(fold_metrics)
    wandb.log({'fold_summary': wandb.Table(dataframe=fold_df)})
    wandb.finish()

LR OOF metrics:
  accuracy: 0.8028
  precision: 1.0000
  recall: 0.0140
  f1: 0.0276
  log_loss: 0.4852
  top1_acc: 0.3825
  map3: 0.5393
  training_time_sec: 4.1771


final/accuracy,▁
final/f1,▁
final/log_loss,▁
final/map3,▁
final/precision,▁
final/recall,▁
final/top1_acc,▁
final/training_time_sec,▁
fold_0/train_accuracy,▁
fold_0/train_f1,▁
+58,...


## Full training and test predictions

In [10]:
clf_full = LogisticRegression(C=1.0, max_iter=1000, solver='liblinear',
                                random_state=SEED)
clf_full.fit(X_tr, y)
test_pred = clf_full.predict_proba(X_te)[:, 1]

np.save(PREDICTIONS_DIR / 'oof_lr.npy', oof_lr)
np.save(PREDICTIONS_DIR / 'test_lr.npy', test_pred)

with open(PREDICTIONS_DIR / 'lr_metrics.json', 'w') as fh:
    json.dump(final_metrics, fh, indent=2)

print(f'saved oof_lr.npy shape={oof_lr.shape}')
print(f'saved test_lr.npy shape={test_pred.shape}')

saved oof_lr.npy shape=(10000,)
saved test_lr.npy shape=(2500,)
